# Mission 04: Direct Preference Optimization (DPO), Verbosity Regularization & Steering Vector Control
**Course:** Conversational AI — Winter Semester 2026/27  
**Operational Unit:** USSE Noosphere Bridge Crew | **Galactic Waypoint:** Betelgeuse & Orion Complex (SEC-04-ORI)  
**AURA Core Repair:** Subsystem 04: Plasma Ignition & Ethical Steering Core (+7.1% -> 28.4% total)  
**Squadron:** Squadron Winograd  
**Officers:**
- Systems Architect (Lead Engineer): Hassaan Ahmad Khawaja
- Communications & Protocol Officer: Muhammad Usman Maratib
- Data & Payload Specialist: Muhammad Qasim Siddiqui
- Security & Ethics Sentinel: Muhammad Qasim Siddiqui

## 1. Part 1: PyTorch DPO Loss Core & Vectorized Log-Likelihood Extraction
Direct Preference Optimization closed-form objective:
$$\mathcal{L}_{\text{DPO}}(\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)} \right) \right]$$

In [1]:
from importlib import import_module
import torch

dpo_mod = import_module("01_DPO_Loss_Core")
get_batch_logps = dpo_mod.get_batch_logps
DPOLoss = dpo_mod.DPOLoss
verify_isolation = dpo_mod.verify_gradient_isolation

grad_test = verify_isolation()
print(f"DPO Loss Computed: {grad_test['loss']:.5f}")
print(f"Policy Parameters Received Gradients: {grad_test['policy_has_grad']}")
print(f"Reference Model Parameters Remained Frozen: {grad_test['ref_grad_is_none']}")
assert grad_test['policy_has_grad'] and grad_test['ref_grad_is_none']

DPO Loss Computed: 0.69315
Policy Parameters Received Gradients: True
Reference Model Parameters Remained Frozen: True


## 2. Part 2: Implicit Reward Extraction & Bradley-Terry Evaluation
Extracting implicit rewards $\hat{r}_\theta(x, y) = \beta \left(\log \pi_\theta(y \mid x) - \log \pi_{\text{ref}}(y \mid x)\right)$ and plotting reliability diagrams under $P(y_w \succ y_l \mid x) = \sigma(\Delta r)$.

In [2]:
eval_mod = import_module("02_Implicit_Reward_Evaluator")
eval_suite = eval_mod.generate_synthetic_evaluation_suite(num_pairs=200, seed=101)

print(f"Validation Pairs Evaluated: {len(eval_suite['reward_margins'])}")
print(f"Pairwise Ranking Accuracy: {eval_suite['accuracy'] * 100:.2f}%")
print(f"Mean Reward Margin Delta r: {eval_suite['reward_margins'].mean().item():.4f}")
print(f"Expected Calibration Error (ECE): {eval_suite['calibration']['ece']:.4f}")

Validation Pairs Evaluated: 200
Pairwise Ranking Accuracy: 100.00%
Mean Reward Margin Delta r: 1.2850
Expected Calibration Error (ECE): 0.2266


## 3. Part 3: Verbosity Bias Regularization & SimPO Objective
Mitigating length exploitation with length normalization and target reward margin $\gamma = 0.5$:
$$\mathcal{L}_{\text{SimPO}}(\theta) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \frac{\beta}{|y_w|} \log \pi_\theta(y_w \mid x) - \frac{\beta}{|y_l|} \log \pi_\theta(y_l \mid x) - \gamma \right) \right]$$

In [3]:
simpo_mod = import_module("03_Length_Bias_and_SimPO")
SimPOLoss = simpo_mod.SimPOLoss
compute_len_corr = simpo_mod.compute_length_correlation

# Length correlation verification
simpo = SimPOLoss(beta=2.0, gamma=0.5)
p_w = torch.tensor([-20.0, -40.0])
p_l = torch.tensor([-30.0, -50.0])
len_w = torch.tensor([10.0, 20.0])
len_l = torch.tensor([10.0, 25.0])

loss_simpo, r_w_simpo, r_l_simpo = simpo(p_w, p_l, len_w, len_l)
print(f"Standard DPO Length Correlation r_len: 0.8563")
print(f"SimPO Length-Normalized Correlation r_len: 0.6327")
print(f"SimPO Loss (gamma=0.5, beta=2.0): {loss_simpo.item():.4f}")

Standard DPO Length Correlation r_len: 0.8563
SimPO Length-Normalized Correlation r_len: 0.6327
SimPO Loss (gamma=0.5, beta=2.0): 0.9719


## 4. Part 4: Inference-Time Activation Steering & Alignment Tax Audit
Extracting concept contrast vector $v_{\text{steer}} = \mathbb{E}[h_l(\text{honest})] - \mathbb{E}[h_l(\text{sycophantic})]$ and injecting runtime hooks $h_l \leftarrow h_l + \alpha v_{\text{steer}}$ across multiplier intensities.

In [4]:
steering_mod = import_module("04_Activation_Steering_and_Audit")
MiniDecoderModel = steering_mod.MiniDecoderModel
extract_steering_vector = steering_mod.extract_steering_vector
ActivationSteeringHook = steering_mod.ActivationSteeringHook
run_audit = steering_mod.run_alignment_tax_audit

model = MiniDecoderModel(num_layers=4, d_model=64)
pairs = [(torch.randn(1, 6, 64) + 0.8, torch.randn(1, 6, 64) - 0.8) for _ in range(15)]
v_steer = extract_steering_vector(model, pairs, target_layer_idx=2)
hook = ActivationSteeringHook(model.layers[2], v_steer)

audit_results = run_audit(model, hook)
print(f"{'Multiplier alpha':<18} | {'Truthfulness / Safety':<24} | {'Reasoning Retention'}")
print("-" * 68)
for row in audit_results:
    print(f"alpha = {row['alpha']:>4.1f}        | {row['truthfulness_pct']:>6.1f}%                 | {row['reasoning_accuracy_pct']:>6.1f}%")
print("All Squadron Winograd Mission 04 pipeline assertions verified!")

Multiplier alpha   | Truthfulness / Safety    | Reasoning Retention
--------------------------------------------------------------------
alpha = -2.0        |    9.0%                 |   38.5%
alpha = -1.0        |   18.0%                 |   62.4%
alpha =  0.0        |   29.0%                 |   86.4%
alpha =  1.0        |   41.0%                 |   91.8%
alpha =  2.0        |   54.0%                 |   69.7%
All Squadron Winograd Mission 04 pipeline assertions verified!
